In [1]:
import qlib
from qlib.config import REG_US # [REG_CN, REG_US]

from qlib.utils import init_instance_by_config
from qlib.utils import flatten_dict
from qlib.workflow import R


import pandas as pd
import numpy as np

import torch
from torch.backends import cuda

import sys
import warnings


# Define directory depth
directory_depth = "../../../../"


# Add custom library to path
sys.path.append(directory_depth + "Custom_Library")
print("Path added successfully: " + directory_depth + "Custom_Library")


# Init Qlib
provider_uri = directory_depth + "Database/Qdata/SPAIN"  # target_dir
qlib.init(provider_uri=provider_uri, region=REG_US)
print("Qlib version: " + str(qlib.__version__))


# GPU
print("GPU enabled: " + str(torch.cuda.is_available()) + "\n")


# Warnings
warnings.filterwarnings("ignore")

Path added successfully: ../../../Custom_Library


[8796:MainThread](2022-03-22 20:14:07,328) INFO - qlib.Initialization - [config.py:405] - default_conf: client.
[8796:MainThread](2022-03-22 20:14:07,331) INFO - qlib.Initialization - [__init__.py:73] - qlib successfully initialized based on client settings.
[8796:MainThread](2022-03-22 20:14:07,333) INFO - qlib.Initialization - [__init__.py:75] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Projects/Investing/Quantitative/Database/Qdata/SPAIN')}


Qlib version: 0.8.3
GPU enabled: True



In [2]:
###################################
# Load artifacts
###################################
import pickle


# Load
with open('artifacts/datahandler.pkl', 'rb') as f:
    defined_datahandler = pickle.load(f)

with open('artifacts/dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

with open('artifacts/model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('artifacts/strategy.pkl', 'rb') as f:
    strategy = pickle.load(f)

In [3]:
# ---------------------------------
# Select start and end time
# ---------------------------------
from datetime import datetime, timedelta
"""
    This helps save time.
"""

start_time = datetime.today() - timedelta(weeks=1)
end_time = None  # datetime(2022, 3, 14)


In [4]:
###################################
# Setup dataset
###################################

# Setup data
defined_datahandler.config()
defined_datahandler.setup_data()


# Init dataset
dataset = {
    "class": "TSDatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "step_len": dataset.step_len,
        "handler": defined_datahandler,
        "segments": {
            "test": (start_time, end_time),
        },
    },
}
dataset = init_instance_by_config(dataset)

[8796:MainThread](2022-03-22 20:14:58,693) INFO - qlib.timer - [log.py:113] - Time cost: 48.686s | Loading data Done
[8796:MainThread](2022-03-22 20:15:01,371) INFO - qlib.timer - [log.py:113] - Time cost: 2.466s | RobustZScoreNorm Done
[8796:MainThread](2022-03-22 20:15:01,499) INFO - qlib.timer - [log.py:113] - Time cost: 0.127s | Fillna Done
[8796:MainThread](2022-03-22 20:15:01,668) INFO - qlib.timer - [log.py:113] - Time cost: 0.106s | DropnaLabel Done
[8796:MainThread](2022-03-22 20:15:12,257) INFO - qlib.timer - [log.py:113] - Time cost: 10.588s | CSZScoreNorm Done
[8796:MainThread](2022-03-22 20:15:12,270) INFO - qlib.timer - [log.py:113] - Time cost: 13.575s | fit & process data Done


In [5]:
###################################
# Setup Account
###################################
from qlib.backtest.account import Account


try:  # Try to load Account
    xlsx_account = pd.ExcelFile('Account.xlsx')

    position_df = pd.read_excel(xlsx_account, sheet_name='Position', index_col="stock_id")
    position_dict = position_df.to_dict(orient='index')

    cash_df = pd.read_excel(xlsx_account, 'Cash')
    cash = cash_df.iloc[0, 0]

    # ------------------------------------------

    trade_account = Account(
        init_cash=cash, position_dict=position_dict, freq="day",
        benchmark_config={
            "benchmark": None,
            "start_time": None,
            "end_time": None,
        }
    )

except FileNotFoundError as e:  # Create new Account
    print("Error: " + str(e) + "\nCreating new Account with 10000 init cash and no stocks held.")

    trade_account = Account(
        init_cash=10000, position_dict={}, freq="day",
        benchmark_config={
            "benchmark": None,
            "start_time": None,
            "end_time": None,
        }
    )

except Exception as e:  # Files exists exists, unknown error
    print(e)


print(trade_account.current_position)

{'_settle_type': None, 'init_cash': 10000, 'position': {'cash': 10000, 'now_account_value': 10000.0}}


In [6]:
###################################
# Setup Exchange
###################################
from qlib.backtest import Exchange


trade_exchange = Exchange(
    start_time=start_time,
    end_time=end_time,
    codes="all",
    deal_price="open",
    open_cost=0,
    close_cost=0,
    min_cost=0,
    extra_quote=None
)

[8796:MainThread](2022-03-22 20:15:28,875) WARNING - qlib.online operator - [exchange.py:205] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.
[8796:MainThread](2022-03-22 20:15:28,876) WARNING - qlib.online operator - [exchange.py:207] - trade unit 1 is not supported in adjusted_price mode.


In [7]:
###################################
# Setup Strategy
###################################
from qlib.backtest.signal import Signal, create_signal_from

from qlib.backtest.utils import CommonInfrastructure

from qlib.backtest.utils import LevelInfrastructure
from qlib.backtest.utils import TradeCalendarManager
import numpy as np
from datetime import datetime, timedelta


# ---------------------------------------------------------------------------------------------------------
# Setup data (such as stop_loss or stock-validation data)
strategy.config()
strategy.setup_data()


# ---------------------------------------------------------------------------------------------------------
# Signal
strategy.signal = create_signal_from((model, dataset))


# ---------------------------------------------------------------------------------------------------------
# CommonInfrastructure
common_infra = CommonInfrastructure(trade_account=trade_account, trade_exchange=trade_exchange)
strategy.reset_common_infra(common_infra)


# ---------------------------------------------------------------------------------------------------------
# LevelInfrastructure
trade_calendar = TradeCalendarManager(freq="day", start_time=start_time, end_time=end_time)
for n in range(2):  # Adding the next 2 time periods to the calendar (next 2 days for this case)
    trade_calendar._calendar = np.append(trade_calendar._calendar, trade_calendar._calendar[-1] + timedelta(days=1))

level_infra = LevelInfrastructure(trade_calendar=trade_calendar)
strategy.reset_level_infra(level_infra)

Stop_Loss basic market data setting up...


[8796:MainThread](2022-03-22 20:15:40,263) INFO - qlib.timer - [log.py:113] - Time cost: 4.510s | Loading data Done


Stop_Loss thresholds data setting up...


[8796:MainThread](2022-03-22 20:15:44,591) INFO - qlib.timer - [log.py:113] - Time cost: 4.322s | Loading data Done
[8796:MainThread](2022-03-22 20:15:54,056) WARNING - qlib.data - [data.py:625] - load calendar error: freq=day, future=True; return current calendar!
[8796:MainThread](2022-03-22 20:15:54,075) WARNING - qlib.data - [data.py:628] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


In [8]:
last_step = strategy.trade_calendar.get_trade_len()
trade_step = last_step

print("###################################################################################################")

pred_start_time, pred_end_time = strategy.trade_calendar.get_step_time(trade_step, shift=1)
print("Predicting period: " + str(pred_start_time) + " / " + str(pred_end_time))

trade_start_time, trade_end_time = strategy.trade_calendar.get_step_time(trade_step)
print("Trading period: " + str(trade_start_time) + " / " + str(trade_end_time))

print("###################################################################################################")

print(strategy.signal.get_signal(pred_start_time, pred_end_time))

###################################################################################################
Predicting period: 2022-03-22 00:00:00 / 2022-03-22 23:59:59
Trading period: 2022-03-23 00:00:00 / 2022-03-23 23:59:59
###################################################################################################
instrument
ACS.MC    -0.176190
ACX.MC     0.245795
AENA.MC   -0.100392
ALM.MC     0.071025
AMS.MC     0.002967
ANA.MC    -0.040814
ANE.MC     0.016285
APPS.MC   -0.084320
BBVA.MC   -0.178391
BKT.MC    -0.074216
CABK.MC   -0.028208
CIE.MC    -0.075277
CLNX.MC   -0.123812
COL.MC    -0.164862
EBRO.MC    0.171918
ELE.MC     0.006967
ENG.MC    -0.169134
FDR.MC     0.121794
FER.MC    -0.135984
GEST.MC    0.115412
GRF.MC     0.360871
IAG.MC    -0.133997
IBE.MC    -0.019451
IDR.MC     0.046083
ITX.MC    -0.082644
LOG.MC     0.033581
MAP.MC    -0.148602
MEL.MC     0.191079
MRL.MC    -0.049114
MTS.MC     0.393278
NTGY.MC   -0.153469
PHM.MC    -0.001588
REE.MC    -0.112987
REP.MC    

In [9]:
###################################
# Generate orders from strategy
###################################
"""
    trade_step is set at the last_step so that generate_trade_decision() produces orders
    for the last available datetime in the calendar, which should be the last datetime with
    trading data in the database.
"""

last_step = strategy.trade_calendar.get_trade_len()
strategy.trade_calendar.trade_step = strategy.trade_calendar.get_trade_len()
print(strategy.trade_calendar)

print("###################################################################################################")

trade_decision = strategy.generate_trade_decision()
orders = trade_decision.get_decision()
print(orders)

class: TradeCalendarManager; 2022-03-15 20:14:09.957155[5717]~None[5721]: [5/5]
###################################################################################################
[Order(stock_id='SLR.MC', amount=83.0, direction=<OrderDir.BUY: 1>, start_time=Timestamp('2022-03-23 00:00:00'), end_time=Timestamp('2022-03-23 23:59:59'), deal_amount=0, factor=None), Order(stock_id='MTS.MC', amount=68.0, direction=<OrderDir.BUY: 1>, start_time=Timestamp('2022-03-23 00:00:00'), end_time=Timestamp('2022-03-23 23:59:59'), deal_amount=0, factor=None), Order(stock_id='SGRE.MC', amount=81.0, direction=<OrderDir.BUY: 1>, start_time=Timestamp('2022-03-23 00:00:00'), end_time=Timestamp('2022-03-23 23:59:59'), deal_amount=0, factor=None), Order(stock_id='GRF.MC', amount=127.0, direction=<OrderDir.BUY: 1>, start_time=Timestamp('2022-03-23 00:00:00'), end_time=Timestamp('2022-03-23 23:59:59'), deal_amount=0, factor=None), Order(stock_id='ACX.MC', amount=194.0, direction=<OrderDir.BUY: 1>, start_time=Ti

In [10]:
###################################
# Parse orders to dataframe
###################################

orders_list = []
for order in orders:
    stock_id = order.stock_id
    amount = order.amount
    start_time = order.start_time
    end_time = order.end_time
    direction = order.direction
    factor = order.factor

    orders_list.append([stock_id, amount, start_time, end_time, direction, factor])


orders_df = pd.DataFrame(
    data=orders_list, columns=['stock_id', 'amount', 'start_time', 'end_time', 'direction', 'factor']
)


orders_df.to_csv("Orders.csv", index=False)

print(orders_df)

  stock_id  amount start_time            end_time  direction factor
0   SLR.MC    83.0 2022-03-23 2022-03-23 23:59:59          1   None
1   MTS.MC    68.0 2022-03-23 2022-03-23 23:59:59          1   None
2  SGRE.MC    81.0 2022-03-23 2022-03-23 23:59:59          1   None
3   GRF.MC   127.0 2022-03-23 2022-03-23 23:59:59          1   None
4   ACX.MC   194.0 2022-03-23 2022-03-23 23:59:59          1   None
5  ROVI.MC     4.0 2022-03-23 2022-03-23 23:59:59          1   None


In [11]:
###################################
# Add stop_loss threshold to order
###################################

sl = strategy.get_stop_loss_df().rename(columns={0: 'stop_loss'})

orders_df = pd.merge(orders_df, sl, how="left", left_on='stock_id', right_index=True)


orders_df.to_csv("Orders.csv", index=False)

print(orders_df)

  stock_id  amount start_time            end_time  direction factor  stop_loss
0   SLR.MC    83.0 2022-03-23 2022-03-23 23:59:59          1   None  18.123215
1   MTS.MC    68.0 2022-03-23 2022-03-23 23:59:59          1   None  25.924643
2  SGRE.MC    81.0 2022-03-23 2022-03-23 23:59:59          1   None  16.845715
3   GRF.MC   127.0 2022-03-23 2022-03-23 23:59:59          1   None  14.372857
4   ACX.MC   194.0 2022-03-23 2022-03-23 23:59:59          1   None   9.009000
5  ROVI.MC     4.0 2022-03-23 2022-03-23 23:59:59          1   None  61.828571
